# Notebook 04 — Trap Curvature and Secular Frequency Proxy

This notebook extends the previous workflow by estimating local curvature of the normalized RF-style pseudopotential near a candidate trapping point.

It focuses on:

- solving Laplace's equation for a simple surface-style electrode layout
- computing the electric field and normalized pseudopotential
- locating a candidate interior trap point
- estimating local curvature using second derivatives
- extracting simple secular-frequency proxies from the local Hessian

This remains a reduced numerical model, but it moves closer to trap characterization and optimization.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Physical note

For a reduced RF-trap model, we use a normalized pseudopotential based on

$$
\Phi_{\mathrm{pseudo}} \propto |E|^2
$$

Near a local minimum, the Hessian of the pseudopotential provides local curvature information. In a full physical model, those curvatures would connect to trap frequencies after including charge, mass, RF frequency, and unit conversions.

Here we compute **dimensionless curvature** and **secular-frequency proxies** from the local Hessian eigenvalues.


## Grid and electrode geometry


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Potential array and fixed-node mask
V = np.zeros((ny, nx), dtype=float)
fixed = np.zeros((ny, nx), dtype=bool)

# Ground outer boundaries
V[:, 0] = 0.0
V[:, -1] = 0.0
V[-1, :] = 0.0
fixed[:, 0] = True
fixed[:, -1] = True
fixed[-1, :] = True

# Lower boundary electrode pattern
bottom = 0
left_dc = (x >= -2.2) & (x <= -0.9)
center_rf = (x >= -0.5) & (x <= 0.5)
right_dc = (x >= 0.9) & (x <= 2.2)
remaining_ground = ~(left_dc | center_rf | right_dc)

V[bottom, left_dc] = -1.0
V[bottom, center_rf] = 1.0
V[bottom, right_dc] = -1.0
V[bottom, remaining_ground] = 0.0
fixed[bottom, :] = True

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')


## Solve Laplace equation


In [ ]:
def idx(i, j, nx):
    return i * nx + j

N = nx * ny
A = lil_matrix((N, N), dtype=float)
b = np.zeros(N, dtype=float)

cx = 1.0 / dx**2
cy = 1.0 / dy**2
c0 = -2.0 * (cx + cy)

for i in range(ny):
    for j in range(nx):
        k = idx(i, j, nx)

        if fixed[i, j]:
            A[k, k] = 1.0
            b[k] = V[i, j]
        else:
            A[k, idx(i, j, nx)] = c0
            A[k, idx(i, j - 1, nx)] = cx
            A[k, idx(i, j + 1, nx)] = cx
            A[k, idx(i - 1, j, nx)] = cy
            A[k, idx(i + 1, j, nx)] = cy

solution = spsolve(A.tocsr(), b)
Vsol = solution.reshape((ny, nx))

print('Laplace solve complete.')
print(f'Potential range: [{Vsol.min():.3f}, {Vsol.max():.3f}]')


## Compute electric field and normalized pseudopotential


In [ ]:
dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
Ex = -dV_dx
Ey = -dV_dy
E_mag = np.sqrt(Ex**2 + Ey**2)
Phi_pseudo = E_mag**2

print(f'Field magnitude range: [{E_mag.min():.3e}, {E_mag.max():.3e}]')
print(f'Normalized pseudopotential range: [{Phi_pseudo.min():.3e}, {Phi_pseudo.max():.3e}]')


## Locate a candidate interior trap point

A naive global minimum search often lands on a flat outer boundary region. For curvature estimation we instead search only in an interior window:

- away from the lower boundary
- away from the outer x boundaries
- away from the top boundary

This produces a more meaningful interior candidate for local Hessian analysis.


In [ ]:
# Interior search margins for a safer candidate point
margin_i_low = 2
margin_i_high = 3
margin_j = 3

i_slice = slice(margin_i_low, ny - margin_i_high)
j_slice = slice(margin_j, nx - margin_j)

Phi_search = Phi_pseudo[i_slice, j_slice]
min_flat = np.argmin(Phi_search)
min_i_local, min_j_local = np.unravel_index(min_flat, Phi_search.shape)

min_i = min_i_local + margin_i_low
min_j = min_j_local + margin_j

trap_x = x[min_j]
trap_y = y[min_i]
trap_value = Phi_pseudo[min_i, min_j]

print(f'Candidate interior trap point: x = {trap_x:.3f}, y = {trap_y:.3f}')
print(f'Pseudopotential at candidate point: {trap_value:.3e}')


## Compute local Hessian at the candidate point

We estimate second derivatives using centered finite differences:

$$
H = \begin{pmatrix}
\partial^2 \Phi / \partial x^2 & \partial^2 \Phi / \partial x \partial y \\
\partial^2 \Phi / \partial y \partial x & \partial^2 \Phi / \partial y^2
\end{pmatrix}
$$

The Hessian eigenvalues describe the principal local curvatures.


In [ ]:
def second_derivatives(F, i, j, dx, dy):
    d2F_dx2 = (F[i, j + 1] - 2.0 * F[i, j] + F[i, j - 1]) / dx**2
    d2F_dy2 = (F[i + 1, j] - 2.0 * F[i, j] + F[i - 1, j]) / dy**2
    d2F_dxdy = (
        F[i + 1, j + 1] - F[i + 1, j - 1] - F[i - 1, j + 1] + F[i - 1, j - 1]
    ) / (4.0 * dx * dy)
    return d2F_dx2, d2F_dy2, d2F_dxdy

d2x, d2y, d2xy = second_derivatives(Phi_pseudo, min_i, min_j, dx, dy)

H = np.array([
    [d2x,  d2xy],
    [d2xy, d2y ]
], dtype=float)

eigvals, eigvecs = np.linalg.eigh(H)

print('Local Hessian:')
print(H)
print('\nPrincipal curvature eigenvalues:')
print(eigvals)


## Local minimum sanity check

We check whether the candidate point is lower than its immediate four neighbors. This is a simple diagnostic, not a full robust classifier.


In [ ]:
center = Phi_pseudo[min_i, min_j]
neighbors = np.array([
    Phi_pseudo[min_i - 1, min_j],
    Phi_pseudo[min_i + 1, min_j],
    Phi_pseudo[min_i, min_j - 1],
    Phi_pseudo[min_i, min_j + 1],
])

is_local_min = np.all(center <= neighbors)
print(f'Immediate-neighbor local-minimum check: {is_local_min}')
print('Neighbor values:')
print(neighbors)


## Secular-frequency proxies

In this reduced notebook, we define secular-frequency proxies from positive curvature eigenvalues as:

$$
\omega_i^{(\mathrm{proxy})} = \sqrt{\max(\lambda_i, 0)}
$$

These are dimensionless curvature-based diagnostics, not yet physical frequencies in SI units.


In [ ]:
omega_proxy = np.sqrt(np.clip(eigvals, 0.0, None))

print('Dimensionless secular-frequency proxies:')
print(omega_proxy)


## Visualize the candidate point on the pseudopotential map


In [ ]:
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, Phi_pseudo, shading='auto')
cs = ax.contour(X, Y, Phi_pseudo, levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.scatter([trap_x], [trap_y], s=80, label='Candidate trap point')
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)
ax.set_title('Normalized RF-Style Pseudopotential with Candidate Trap Point')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
fig.colorbar(im, ax=ax, label='Pseudo')
plt.tight_layout()
plt.show()


## Local cuts near the candidate point

We inspect the pseudopotential near the candidate point to see whether the local structure looks trap-like.


In [ ]:
j_window = slice(max(min_j - 8, 0), min(min_j + 9, nx))
i_window = slice(max(min_i - 8, 0), min(min_i + 9, ny))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x[j_window], Phi_pseudo[min_i, j_window], label='Local x-cut')
ax.axvline(trap_x, linestyle='--', linewidth=1)
ax.set_title('Local Pseudopotential Cut Along x')
ax.set_xlabel('x')
ax.set_ylabel('Pseudo')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y[i_window], Phi_pseudo[i_window, min_j], label='Local y-cut')
ax.axvline(trap_y, linestyle='--', linewidth=1)
ax.set_title('Local Pseudopotential Cut Along y')
ax.set_xlabel('y')
ax.set_ylabel('Pseudo')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Notes and next steps

This notebook adds local curvature diagnostics to the reduced pseudopotential workflow.

Natural next steps:

- include physical constants to convert curvature to SI-scale secular frequencies
- compare multiple electrode geometries
- add DC compensation fields and axial shaping
- reject weak or non-confining minima automatically
- sweep voltages and electrode widths to map trap strength
- optimize for trap height, curvature balance, and robustness

That progression will move the workflow toward practical ion-trap parameter optimization.
